##Huggingface Setup

In [ ]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

Looking in indexes: https://download.pytorch.org/whl/cu121


In [ ]:
!huggingface-cli login


    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? (Y/n) y
Token is valid (permission: fineGrained).
The token `colab` has been saved to /root/.cache/huggingface/stored_tokens
Cannot authenticate through git-credential as no helper is defined on your machine.
You might have to re-authenticate wh

In [ ]:
import transformers
print(transformers.__version__)

4.53.0


In [ ]:
!pip install -U transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 87.2 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 4.52.4
    Uninstalling transformers-4.52.4:
      Successfully uninstalled transformers-4.52.4


In [ ]:
!pip install timm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 33.9 MB/s eta 0:00:00


In [ ]:
from transformers import AutoProcessor, Gemma3nForConditionalGeneration
from PIL import Image
import requests
import torch

model_id = "google/gemma-3n-e2b-it"

model = Gemma3nForConditionalGeneration.from_pretrained(model_id, torch_dtype=torch.bfloat16,).eval()

#model = Gemma3nForConditionalGeneration.from_pretrained(model_id, torch_dtype=torch.bfloat16,).eval().to("cuda")

processor = AutoProcessor.from_pretrained(model_id)

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

processor_config.json:   0%|          | 0.00/98.0 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/1.63k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/1.12k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.20M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.70M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/769 [00:00<?, ?B/s]

In [ ]:
messages = [
    {
        "role": "system",
        "content": [{"type": "text", "text": "You are a helpful assistant."}]
    },
    {
        "role": "user",
        "content": [
            {"type": "image", "image": "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/bee.jpg"},
            {"type": "text", "text": "Describe this image in detail."}
        ]
    }
]

inputs = processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device, dtype=torch.bfloat16)

input_len = inputs["input_ids"].shape[-1]

with torch.inference_mode():
    generation = model.generate(**inputs, max_new_tokens=100, do_sample=False)
    generation = generation[0][input_len:]

decoded = processor.decode(generation, skip_special_tokens=True)
print(decoded)

The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The image shows a close-up view of a vibrant pink cosmos flower in full bloom, with a small bee diligently collecting pollen from its center. The flower has delicate, slightly ruffled petals that radiate outwards from a bright yellow center. 

The bee is a dark color with lighter stripes on its abdomen, and it appears to be actively foraging on the flower's reproductive parts. 

The background is softly blurred, suggesting a garden setting with other flowers and foliage. There are hints of other pink


In [ ]:
from PIL import Image
image = Image.open("/content/1_GZ2oHiedOWDoqTZV6ady3g-ezgif.com-webp-to-jpg-converter.jpg").convert("RGB")
prompt = "What is in the image?"

In [ ]:
messages = [
    {
        "role": "system",
        "content": [{"type": "text", "text": "You are a helpful assistant."}]
    },
    {
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": prompt}
        ]
    }
]

inputs = processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device, dtype=torch.bfloat16)

input_len = inputs["input_ids"].shape[-1]

with torch.inference_mode():
    generation = model.generate(**inputs, max_new_tokens=1000, do_sample=False)
    generation = generation[0][input_len:]

decoded = processor.decode(generation, skip_special_tokens=True)
print(decoded)

In [ ]:
from PIL import Image
image = Image.open("/content/basic-invoice-template.png").convert("RGB")
prompt = "Extract text from image?"

messages = [
    {
        "role": "system",
        "content": [{"type": "text", "text": "You are a helpful assistant."}]
    },
    {
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": prompt}
        ]
    }
]

inputs = processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device, dtype=torch.bfloat16)

input_len = inputs["input_ids"].shape[-1]

with torch.inference_mode():
    generation = model.generate(**inputs, max_new_tokens=1000, do_sample=False)
    generation = generation[0][input_len:]

decoded = processor.decode(generation, skip_special_tokens=True)
print(decoded)

The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Here's the extracted text from the image:

**Invoice**

**Company Name:** [Company Name]
**Address:** [Address]
**Phone:** [Phone Number]
**Email:** [Email Address]

**Invoice Number:** [Invoice Number]
**Date:** [Date]

**Bill To:**
[Customer Name]
[Customer Address]
[Customer Phone]
[Customer Email]

**Items:**

| Item | Quantity | Unit Price | Amount |
|---|---|---|---|
| [Item Description 1] | [Quantity 1] | [Unit Price 1] | [Amount 1] |
| [Item Description 2] | [Quantity 2] | [Unit Price 2] | [Amount 2] |
| [Item Description 3] | [Quantity 3] | [Unit Price 3] | [Amount 3] |
| ... | ... | ... | ... |

**Total:** [Total Amount]

**Thank you for your order!**

**Note:** The image contains a partially filled invoice. Some fields are blank and require further information.


In [ ]:
import torchaudio
from transformers import AutoProcessor, Speech2TextForConditionalGeneration
from pathlib import Path
import torch

audio_path = "/content/monsur blog.mp3"
waveform, sample_rate = torchaudio.load(audio_path)

if sample_rate != 16000:
    waveform = torchaudio.transforms.Resample(orig_freq=sample_rate, new_freq=16000)(waveform)
    sample_rate = 16000

prompt = "Transcribe the following speech segment in English, then translate it into german, bangla and spanish:"

messages = [
    {
        "role": "system",
        "content": [{"type": "text", "text": "You are a helpful assistant."}]
    },
    {
        "role": "user",
        "content": [
            {"type": "audio", "audio": waveform.squeeze(0).numpy()},
            {"type": "text", "text": prompt}
        ]
    }
]

inputs = processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device, dtype=torch.bfloat16)

input_len = inputs["input_ids"].shape[-1]

with torch.inference_mode():
    generation = model.generate(**inputs, max_new_tokens=1000, do_sample=False)
    generation = generation[0][input_len:]

decoded = processor.decode(generation, skip_special_tokens=True)
print(decoded)

In [ ]:
prompt = "Three different numbers add up to twelve. The sum of the reciprocal of the first and the product of the other two is also twelve — what is the product of all three numbers?"

messages = [
    {
        "role": "system",
        "content": [{"type": "text", "text": "You are a helpful assistant."}]
    },
    {
        "role": "user",
        "content": [
            {"type": "text", "text": prompt}
        ]
    }
]

inputs = processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device, dtype=torch.bfloat16)

input_len = inputs["input_ids"].shape[-1]

with torch.inference_mode():
    generation = model.generate(**inputs, max_new_tokens=1000, do_sample=False)
    generation = generation[0][input_len:]

decoded = processor.decode(generation, skip_special_tokens=True)
print(decoded)

The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


In [ ]:
prompt = """You are given three distinct positive integers that add up to a given number S.

Write a function that finds all possible triplets (a, b, c) such that:

- a + b + c == S
- 1/a + (b * c) == S  (i.e., the reciprocal of the first plus the product of the other two equals the same total)

Return all valid triplets as a list of tuples, sorted in ascending order of a.

Example Input:
S = 12
Example Output:
[(3, 4, 5)]
"""


messages = [
    {
        "role": "system",
        "content": [{"type": "text", "text": "You are a helpful assistant."}]
    },
    {
        "role": "user",
        "content": [
            {"type": "text", "text": prompt}
        ]
    }
]

inputs = processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device, dtype=torch.bfloat16)

input_len = inputs["input_ids"].shape[-1]

with torch.inference_mode():
    generation = model.generate(**inputs, max_new_tokens=1000, do_sample=False)
    generation = generation[0][input_len:]

decoded = processor.decode(generation, skip_special_tokens=True)
print(decoded)

## Ollama setup

In [1]:
!sudo apt update
!sudo apt install -y pciutils
!curl -fsSL https://ollama.com/install.sh | sh

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:3 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [1,801 kB]
Get:6 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,067 kB]
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:10 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,257 kB]
Get:11 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease [24.3 kB]
Get:12 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Hit:13 https://ppa.launchp

In [4]:
!pip install ollama

In [2]:
import threading
import subprocess
import time

def run_ollama_serve():
    subprocess.Popen(["ollama", "serve"])

thread = threading.Thread(target=run_ollama_serve)
thread.start()
time.sleep(5)

In [3]:
!ollama pull gemma3n:e4b

In [5]:
!ollama list

NAME           ID              SIZE      MODIFIED      
gemma3n:e4b    15cb39fd9394    7.5 GB    7 seconds ago    


In [6]:
from ollama import Client

client = Client()  # Connect to local Ollama server

prompt = """You are given three distinct positive integers that add up to a given number S.

Write a function that finds all possible triplets (a, b, c) such that:

- a + b + c == S
- 1/a + (b * c) == S  (i.e., the reciprocal of the first plus the product of the other two equals the same total)

Return all valid triplets as a list of tuples, sorted in ascending order of a.

Example Input:
S = 12
Example Output:
[(3, 4, 5)]
"""

response = client.chat(
    model='gemma3n:e4b',
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": prompt}
    ]
)

print(response['message']['content'])


```python
def find_triplets(S):
    """
    Finds all possible triplets (a, b, c) of distinct positive integers that satisfy the given conditions.

    Args:
        S: The target sum.

    Returns:
        A list of tuples, where each tuple represents a valid triplet (a, b, c), sorted in ascending order of a.
    """

    triplets = []
    for a in range(1, S // 3 + 1):  # Iterate through possible values of 'a'
        for b in range(a + 1, (S - a) // 2 + 1):  # Iterate through possible values of 'b' (must be greater than 'a')
            c = S - a - b  # Calculate 'c' based on 'a' and 'b'
            if c > b and 1 / a + (b * c) == S:  # Check if 'c' is greater than 'b' and if the equation holds
                triplets.append(tuple(sorted((a, b, c))))  # Add the triplet to the list (sorted for consistency)

    return sorted(list(set(triplets)))  # Remove duplicates and sort the list


# Example usage:
S = 12
result = find_triplets(S)
print(result)  # Output: [(3, 4, 5)]

S = 10
res

In [7]:
from ollama import Client

client = Client()

prompt = "Three different numbers add up to twelve. The sum of the reciprocal of the first and the product of the other two is also twelve — what is the product of all three numbers?"

response = client.chat(
    model='gemma3n:e4b',
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": prompt}
    ]
)

print(response['message']['content'])


Let the three different numbers be $a, b, c$. We are given that
$$a + b + c = 12 \quad (*)$$
and
$$\frac{1}{a} + bc = 12 \quad (**)$$
From $(*)$, we have $a = 12 - b - c$. Substituting this into $(**)$, we get
$$\frac{1}{12 - b - c} + bc = 12$$
$$\frac{1}{12 - b - c} = 12 - bc$$
$$1 = (12 - b - c)(12 - bc)$$
$$1 = 144 - 12bc - 12b + b^2c - 12c + bc^2 + bc^2 - b^2c$$
$$1 = 144 - 12bc - 12b - 12c + b^2c + bc^2$$
$$1 = 144 - 12(bc + b + c) + bc(b+c)$$
$$1 - 144 = -12(bc + b + c) + bc(b+c)$$
$$-143 = -12(bc + b + c) + bc(b+c)$$
Let $bc + b + c = x$. Then $b+c = x - bc$.
Substituting this into the equation, we have
$$-143 = -12x + bc(x-bc)$$
$$-143 = -12x + bcx - (bc)^2$$
$$(bc)^2 - bcx - 12x - 143 = 0$$
Let $P = abc$. We want to find $P$.
From $a + b + c = 12$, we have $a = 12 - b - c$.
From $\frac{1}{a} + bc = 12$, we have $\frac{1}{12 - b - c} + bc = 12$.
Multiplying by $12 - b - c$, we get $1 + bc(12 - b - c) = 12(12 - b - c)$.
$1 + 12bc - b^2c - bc^2 = 144 - 12b - 12c$
$12bc - b^2c - b